In [ ]:

from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# PharmaOpt project folders
PROJECT_DIR = Path.home() / "PharmaOpt"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DASHBOARD_DIR = OUTPUT_DIR / "interactive_dashboard"

DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

print("PharmaOpt Interactive Dashboard")
print("Project folder:", PROJECT_DIR)
print("Dashboard output folder:", DASHBOARD_DIR)
print("Plotly is ready!")


In [ ]:

# CELL 2 — Locate and inspect the PharmaOpt dataset

DATA_FILE = OUTPUT_DIR / "pharmaopt_integrated_package_year_v05.csv"

if DATA_FILE.exists():
    print("Dataset found!")
    print("File:", DATA_FILE)

    # Read just 5 rows first to inspect the columns
    preview = pd.read_csv(DATA_FILE, nrows=5, low_memory=False)

    print("\nDataset columns:")
    print(preview.columns.tolist())

    display(preview.head())
else:
    print("Dataset not found at:", DATA_FILE)
    print("\nAvailable CSV files in outputs:")

    for file in OUTPUT_DIR.rglob("*.csv"):
        print(file.relative_to(OUTPUT_DIR))


In [ ]:

# CELL 3 — Identify year and shortage-label columns

print("Potential year columns:")
year_columns = [
    col for col in preview.columns
    if "year" in col.lower()
    or "prediction_date" in col.lower()
]

print(year_columns)

print("\nPotential shortage-label columns:")
target_columns = [
    col for col in preview.columns
    if any(word in col.lower()
           for word in ["target", "shortage", "label"])
]

print(target_columns)


In [ ]:

# CELL 4 — Interactive shortage labels by year

# Load only the two columns we need
df = pd.read_csv(
    DATA_FILE,
    usecols=["prediction_year", "shortage_target"]
)

# Summarize observations and shortage labels
yearly = (
    df.groupby("prediction_year", as_index=False)
    .agg(
        total_packages=("shortage_target", "size"),
        shortage_labels=("shortage_target", "sum")
    )
)

yearly["shortage_rate"] = (
    yearly["shortage_labels"] / yearly["total_packages"] * 100
)

display(yearly)

# Create interactive chart
fig = px.bar(
    yearly,
    x="prediction_year",
    y="shortage_labels",
    text="shortage_labels",
    custom_data=["total_packages", "shortage_rate"],
    title="PharmaOpt | FDA-Linked Shortage Labels by Year",
    labels={
        "prediction_year": "Prediction Year",
        "shortage_labels": "Positive Shortage Labels"
    },
    color="shortage_labels",
    color_continuous_scale="Teal"
)

fig.update_traces(
    hovertemplate=(
        "<b>Year: %{x}</b><br>"
        "Positive labels: %{y:,}<br>"
        "Total package-year observations: %{customdata[0]:,}<br>"
        "Positive label rate: %{customdata[1]:.3f}%"
        "<extra></extra>"
    ),
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=500,
    showlegend=False,
    coloraxis_showscale=False,
    xaxis=dict(dtick=1),
    yaxis_title="Number of Positive Labels"
)

fig.show()


In [ ]:

# CELL 5 — Data quality checks

import numpy as np

required = {"prediction_year", "shortage_target"}
assert required.issubset(df.columns), "Run Cell 4 first."

quality = df.copy()
quality["prediction_year"] = pd.to_numeric(
    quality["prediction_year"], errors="coerce"
)
quality["shortage_target"] = pd.to_numeric(
    quality["shortage_target"], errors="coerce"
)

invalid_labels = (
    quality["shortage_target"].notna()
    & ~quality["shortage_target"].isin([0, 1])
)

print("PHARMAOPT DATA QUALITY CHECK")
print("-" * 45)
print(f"Total package-year rows: {len(quality):,}")
print(f"Missing years: {quality['prediction_year'].isna().sum():,}")
print(f"Missing shortage labels: {quality['shortage_target'].isna().sum():,}")
print(f"Invalid shortage labels: {invalid_labels.sum():,}")
print(f"Positive shortage labels: {(quality['shortage_target'] == 1).sum():,}")

year_summary = (
    quality.groupby("prediction_year", dropna=False)
    .agg(
        package_years=("shortage_target", "size"),
        positive_labels=("shortage_target", lambda x: (x == 1).sum()),
        missing_labels=("shortage_target", lambda x: x.isna().sum())
    )
    .reset_index()
)

year_summary["positive_rate_pct"] = (
    year_summary["positive_labels"]
    / year_summary["package_years"] * 100
)

display(year_summary)


In [ ]:

# CELL 6 — Positive label rate by year

fig_rate = px.line(
    year_summary.dropna(subset=["prediction_year"]),
    x="prediction_year",
    y="positive_rate_pct",
    markers=True,
    text="positive_rate_pct",
    custom_data=["package_years", "positive_labels"],
    title="PharmaOpt | FDA-Linked Positive Label Rate",
    labels={
        "prediction_year": "Prediction Year",
        "positive_rate_pct": "Positive Label Rate (%)"
    }
)

fig_rate.update_traces(
    line=dict(color="#087F8C", width=3),
    marker=dict(size=12),
    texttemplate="%{y:.4f}%",
    textposition="top center",
    hovertemplate=(
        "Year: %{x:.0f}<br>"
        "Label rate: %{y:.4f}%<br>"
        "Package-years: %{customdata[0]:,}<br>"
        "Positive labels: %{customdata[1]:,}"
        "<extra></extra>"
    )
)

fig_rate.update_layout(
    template="plotly_white",
    height=480,
    xaxis=dict(dtick=1),
    yaxis=dict(rangemode="tozero")
)

fig_rate.show()


In [ ]:

# CELL 7 — Package-year dataset growth

fig_growth = px.bar(
    year_summary.dropna(subset=["prediction_year"]),
    x="prediction_year",
    y="package_years",
    text="package_years",
    title="PharmaOpt | Package-Year Dataset Coverage",
    labels={
        "prediction_year": "Prediction Year",
        "package_years": "Package-Year Observations"
    }
)

fig_growth.update_traces(
    marker_color="#173A5E",
    texttemplate="%{y:,.0f}",
    textposition="outside",
    hovertemplate=(
        "Year: %{x:.0f}<br>"
        "Package-year observations: %{y:,}"
        "<extra></extra>"
    )
)

fig_growth.update_layout(
    template="plotly_white",
    height=480,
    xaxis=dict(dtick=1),
    yaxis=dict(rangemode="tozero")
)

fig_growth.show()


In [ ]:

# CELL 8 — Complete data-quality audit and interactive chart

audit_columns = [
    "package_ndc",
    "product_ndc",
    "generic_name",
    "labeler_name",
    "dosage_form",
    "marketing_category",
    "prediction_year",
    "shortage_target"
]

# Read identifiers as text to preserve leading zeros
audit_data = pd.read_csv(
    DATA_FILE,
    usecols=audit_columns,
    dtype={
        "package_ndc": "string",
        "product_ndc": "string"
    },
    low_memory=False
)

# Count missing and blank values
missing_rows = []

for col in audit_columns:
    series = audit_data[col]
    missing = int(series.isna().sum())

    if pd.api.types.is_string_dtype(series) or series.dtype == "object":
        blank = int(
            series.astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
            .sum()
        )
        missing += blank

    missing_rows.append({
        "field": col,
        "missing_rows": missing,
        "missing_pct": missing / len(audit_data) * 100
    })

missing_summary = (
    pd.DataFrame(missing_rows)
    .sort_values("missing_pct", ascending=False)
)

# Check duplicate package-year identifiers
duplicate_keys = int(
    audit_data.duplicated(
        subset=["package_ndc", "prediction_year"]
    ).sum()
)

print("PHARMAOPT IDENTIFIER AUDIT")
print("-" * 45)
print(f"Rows checked: {len(audit_data):,}")
print(f"Duplicate package-year keys: {duplicate_keys:,}")

display(missing_summary)

# Interactive chart
fig_missing = px.bar(
    missing_summary,
    x="missing_pct",
    y="field",
    orientation="h",
    text="missing_rows",
    custom_data=["missing_rows"],
    title="PharmaOpt | Missing Data by Field",
    labels={
        "missing_pct": "Missing Data (%)",
        "field": "Data Field"
    }
)

fig_missing.update_traces(
    marker_color="#C18B37",
    texttemplate="%{text} missing",
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Missing rows: %{customdata[0]:,}<br>"
        "Missing percentage: %{x:.5f}%"
        "<extra></extra>"
    )
)

fig_missing.update_layout(
    template="plotly_white",
    height=500,
    yaxis=dict(autorange="reversed"),
    xaxis=dict(rangemode="tozero")
)

fig_missing.show()


In [ ]:

# CELL 9 — Dosage form distribution

dosage_summary = (
    audit_data["dosage_form"]
    .fillna("Unknown")
    .value_counts()
    .head(12)
    .reset_index()
)

dosage_summary.columns = ["dosage_form", "package_years"]

fig_dosage = px.bar(
    dosage_summary.sort_values("package_years"),
    x="package_years",
    y="dosage_form",
    orientation="h",
    title="PharmaOpt | Most Common Dosage Forms",
    labels={
        "package_years": "Package-Year Observations",
        "dosage_form": "Dosage Form"
    }
)

fig_dosage.update_traces(
    marker_color="#087F8C",
    hovertemplate=(
        "%{y}<br>"
        "Package-year observations: %{x:,}"
        "<extra></extra>"
    )
)

fig_dosage.update_layout(
    template="plotly_white",
    height=570
)

fig_dosage.show()

print(
    "Note: These are package-year observations, "
    "not counts of unique medications."
)


In [ ]:

# CELL 10 — Preliminary model comparison

model_results = pd.DataFrame({
    "model": ["v0.4 Baseline", "v0.5 Recall Features"],
    "pr_auc": [0.003133, 0.002396],
    "roc_auc": [0.776949, 0.759436],
    "positives_in_top_500": [10, 6]
})

display(model_results)

fig_model = px.bar(
    model_results,
    x="model",
    y="pr_auc",
    text="pr_auc",
    title="PharmaOpt | Preliminary Model Comparison: PR-AUC",
    labels={
        "model": "Model Version",
        "pr_auc": "Precision-Recall AUC"
    }
)

fig_model.update_traces(
    marker_color=["#173A5E", "#087F8C"],
    texttemplate="%{y:.6f}",
    textposition="outside"
)

fig_model.update_layout(
    template="plotly_white",
    height=480,
    yaxis=dict(rangemode="tozero"),
    showlegend=False
)

fig_model.show()

print(
    "Interpretation: v0.4 had the higher PR-AUC "
    "on the previously examined 2025 holdout."
)
print(
    "Important: The holdout has been examined repeatedly, "
    "and historical shortage labels require further validation."
)


In [ ]:

# CELL 11 — Export PharmaOpt dashboard and audit tables

import plotly.io as pio
from html import escape

charts = [
    ("FDA-Linked Shortage Labels", fig),
    ("Positive Label Rate", fig_rate),
    ("Dataset Coverage", fig_growth),
    ("Missing Data", fig_missing),
    ("Dosage Form Distribution", fig_dosage),
    ("Preliminary Model Comparison", fig_model)
]

html_sections = []

for i, (heading, chart) in enumerate(charts):
    chart_html = pio.to_html(
        chart,
        full_html=False,
        include_plotlyjs=True if i == 0 else False
    )

    html_sections.append(
        f"<section><h2>{escape(heading)}</h2>"
        f"{chart_html}</section>"
    )

dashboard_html = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>PharmaOpt | Interactive Analytics</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f3f6f9;
    color: #173A5E;
    max-width: 1100px;
    margin: 0 auto;
    padding: 30px;
}
header {
    background: #173A5E;
    color: white;
    padding: 28px;
    border-radius: 12px;
}
section {
    background: white;
    padding: 16px;
    margin-top: 24px;
    border-radius: 12px;
    box-shadow: 0 2px 10px #00000012;
}
.note {
    margin-top: 24px;
    padding: 16px;
    background: #fff8e9;
    border-radius: 8px;
}
</style>
</head>
<body>
<header>
<h1>PharmaOpt</h1>
<p>Pharmaceutical Supply-Chain Analytics</p>
<p>Interactive Data Quality & Model Exploration</p>
</header>
""" + "\n".join(html_sections) + """
<div class="note">
<strong>Research limitations:</strong>
FDA-linked shortage labels do not represent all
national drug shortages. Historical coverage and
status definitions remain under review.
Model comparisons are preliminary, and the
2025 holdout is not an untouched final test.
</div>
</body>
</html>
"""

dashboard_file = (
    DASHBOARD_DIR / "PharmaOpt_Interactive_Dashboard.html"
)

dashboard_file.write_text(
    dashboard_html,
    encoding="utf-8"
)

year_summary.to_csv(
    DASHBOARD_DIR / "yearly_label_summary.csv",
    index=False
)

missing_summary.to_csv(
    DASHBOARD_DIR / "missing_data_summary.csv",
    index=False
)

model_results.to_csv(
    DASHBOARD_DIR / "preliminary_model_results.csv",
    index=False
)

print("Dashboard saved successfully!")
print(dashboard_file)
print("You can open this HTML file in Chrome or Edge.")


In [ ]:

# CELL 12 — Open the dashboard from JupyterLab

from IPython.display import display, FileLink

print("Your interactive PharmaOpt dashboard is ready:")
display(FileLink(str(dashboard_file)))


In [ ]:

# Find exact duplicate files in PharmaOpt (read-only)

from pathlib import Path
from collections import defaultdict
import hashlib

root = Path.home() / "PharmaOpt"
groups = defaultdict(list)

for path in root.rglob("*"):
    if not path.is_file():
        continue
    if ".ipynb_checkpoints" in path.parts:
        continue

    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    groups[(path.stat().st_size, digest.hexdigest())].append(path)

duplicates = [
    files for files in groups.values()
    if len(files) > 1
]

if duplicates:
    print("EXACT DUPLICATE FILES:\n")
    for group in duplicates:
        for file in group:
            print(file.relative_to(root))
        print("-" * 40)
else:
    print("No exact duplicate files found.")

print(f"\nDuplicate groups: {len(duplicates)}")


In [ ]:

# CELL 13 — Inspect CMS source ZIP files (read-only)

from pathlib import Path
import zipfile
import hashlib

root = Path.home() / "PharmaOpt"

files = [
    root / "cms_partd_historical_2019.zip",
    root / "cms_partd_historical_2020.zip"
]

for file in files:
    print(f"\nFILE: {file.name}")

    if not file.exists():
        print("File not found")
        continue

    print(f"Size: {file.stat().st_size:,} bytes")

    with zipfile.ZipFile(file, "r") as z:
        for info in z.infolist():
            print(f"Inside ZIP: {info.filename}")
            print(f"Uncompressed size: {info.file_size:,} bytes")

    with open(file, "rb") as f:
        digest = hashlib.file_digest(f, "sha256").hexdigest()

    print(f"SHA-256: {digest}")

print("\nInspection complete. No files were modified.")


In [ ]:

# CELL 14 — Verify CMS source years and CSV structure

import zipfile
import pandas as pd
from pathlib import Path

cms_zip = Path.home() / "PharmaOpt" / "cms_partd_historical_2019.zip"

with zipfile.ZipFile(cms_zip, "r") as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
    ]

    for name in sorted(csv_files):
        print("\n" + "=" * 65)
        print("CMS FILE:", name)

        with z.open(name) as f:
            preview_cms = pd.read_csv(f, nrows=3, low_memory=False)

        print("Columns:", preview_cms.columns.tolist())
        display(preview_cms.head(3))

print("\nCMS source inspection complete.")
